# Multi-objective optimization (NSGA-II) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Rank by Pareto dominance, then preserve spread with crowding distance
NSGA-II handles objectives that conflict. Instead of collapsing them into one arbitrary score, it sorts candidates into Pareto fronts and uses crowding distance so the surviving set covers the tradeoff curve rather than crowding one corner.

In [ ]:
# 1) Non-dominated sorting finds the first Pareto front
pts=np.array([[1,5],[2,3],[3,2],[5,1],[3,4],[4,3]],float)
front=[]
for i,p in enumerate(pts):
    dominated=any(np.all(q<=p) and np.any(q<p) for j,q in enumerate(pts) if j!=i)
    if not dominated: front.append(i)
colors=['tab:green' if i in front else 'tab:red' for i in range(len(pts))]
plt.figure(figsize=(4,4)); plt.scatter(pts[:,0],pts[:,1],c=colors,s=90); plt.xlabel('objective 1'); plt.ylabel('objective 2'); plt.title('green points are not dominated')
assert front==[0,1,2,3]

In [ ]:
# 2) Crowding distance protects the spread along a front
front_pts=np.array([[1.,5.],[2.,3.],[3.,2.],[5.,1.]]); d=np.zeros(len(front_pts))
for k in range(2):
    order=np.argsort(front_pts[:,k]); d[order[0]]=d[order[-1]]=np.inf; rng=front_pts[order[-1],k]-front_pts[order[0],k]
    for r in range(1,len(front_pts)-1): d[order[r]] += (front_pts[order[r+1],k]-front_pts[order[r-1],k])/rng
plt.figure(figsize=(6,3)); plt.bar([str(tuple(p)) for p in front_pts], np.where(np.isinf(d),2.0,d)); plt.ylabel('crowding distance (inf clipped)'); plt.title('boundary and sparse points are protected')
assert np.isinf(d[0]) and np.isinf(d[-1]) and np.allclose(d[1:3],[1.25,1.25])

In [ ]:
# 3) Tournament selection prefers lower rank, then higher crowding
rank=np.array([0,0,1,0]); crowd=np.array([0.4,1.2,5.0,0.8]); a,b=1,2
winner=a if (rank[a]<rank[b] or (rank[a]==rank[b] and crowd[a]>crowd[b])) else b
plt.figure(figsize=(6,3)); plt.scatter(rank,crowd,s=100); plt.scatter([rank[winner]],[crowd[winner]],c='r',s=160); plt.xlabel('rank (lower better)'); plt.ylabel('crowding (higher better)'); plt.title('rank beats crowding in NSGA-II tournaments')
assert winner==1

In [ ]:
# 4) Keeping a diverse Pareto set beats scalarizing to one point
x=np.linspace(0,1,80); objs=np.c_[x**2,(1-x)**2]; score=objs.sum(axis=1); chosen_scalar=objs[np.argmin(score)]; chosen_pareto=objs[::15]
plt.figure(figsize=(4,4)); plt.plot(objs[:,0],objs[:,1]); plt.scatter([chosen_scalar[0]],[chosen_scalar[1]],c='r',label='single scalar best'); plt.scatter(chosen_pareto[:,0],chosen_pareto[:,1],c='g',label='Pareto spread'); plt.legend(); plt.xlabel('f1'); plt.ylabel('f2'); plt.title('NSGA-II returns a front, not one compromise')
assert chosen_pareto.shape[0] > 1 and abs(chosen_scalar[0]-chosen_scalar[1]) < 0.02

In [ ]:
# 5) A tiny evolutionary loop improves Pareto coverage
rng=np.random.default_rng(9); pop=rng.uniform(0,1,30); history=[]
def objs1(z): return np.c_[z**2,(1-z)**2]
for _ in range(18):
    o=objs1(pop); front=np.ones(len(pop),dtype=bool)
    for i,p in enumerate(o): front[i]=not any(np.all(q<=p) and np.any(q<p) for j,q in enumerate(o) if j!=i)
    history.append(np.ptp(pop[front]) if front.any() else 0)
    parents=rng.choice(pop,20); children=np.clip(parents+rng.normal(0,0.08,20),0,1); pop=np.r_[pop,children]
    # truncate by even spread in decision space for this toy front
    pop=np.sort(pop)[np.linspace(0,len(pop)-1,30).astype(int)]
plt.figure(figsize=(6,3)); plt.plot(history,'-o'); plt.ylabel('front x-range'); plt.xlabel('generation'); plt.title('coverage of the tradeoff remains broad')
assert history[-1] > 0.8